In [1]:
from rdkit import Chem
import json, random
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
import multiprocessing
from functools import partial
import sys
from sltools import property_tools

In [2]:
def create_prompt(sm_str):
    template = {
    "instruction": "You love and excel at generating SMILES strings of drug-like molecules",
    "input":None,
    "output": None
    }
    mol = Chem.MolFromSmiles(sm_str)
    if not mol:
         return {}
    else:
        kwargs = {'hbd_range': (0.5 > random.random())}
        kwargs['hba_range'] =  (0.5 > random.random())
        kwargs['mw_range'] =  (0.5 > random.random())
        kwargs['logp_range'] =  (0.5 > random.random())
        kwargs['rotb_range'] =  (0.5 > random.random())
        kwargs['fracsp3_range'] =  (0.5 > random.random())
        kwargs['tpsa_range'] =  (0.5 > random.random())
        kwargs['macrocycle'] =  (0.5 > random.random())
        kwargs['formula'] =  (0.5 > random.random())
        kwargs['undesirable_smarts'] = (0.5 > random.random())
        kwargs['cov_warhead'] = (0.5 > random.random())
        kwargs['substruct'] = (0.5 > random.random())
    #print(kwargs)
    input = property_tools.generate_inp_prompt(mol, **kwargs)
    return {**template, "output":sm_str, "input":input}

def parallelize_processing(sm_strs):
    sm_strs = list(set(sm_strs))
    # Determine the number of CPU cores to use
    num_cores = multiprocessing.cpu_count()
    # Create a pool of workers
    with multiprocessing.Pool(processes=num_cores) as pool:
        results = list(tqdm(pool.imap(create_prompt, sm_strs), total=len(sm_strs)))
    return results

def convert_smiles_list(sm_strs, doRandom, canonical, kekuleSmiles, allHsExplicit):
    # Determine the number of CPU cores to use
    num_cores = multiprocessing.cpu_count()
    print("Number of cores: ", num_cores)

    # Create argument tuples for each SMILES string
    args = [(sm_str, doRandom, canonical, kekuleSmiles, allHsExplicit)
            for sm_str in sm_strs]

    with multiprocessing.Pool(processes=num_cores) as pool:
        results = list(tqdm(pool.imap(convert_smiles_string, args),
                          total=len(sm_strs)))

    # Remove None values (failed conversions)
    results = [r for r in results if r is not None]
    return results

def convert_smiles_string(args):
    # Unpack all arguments from the tuple
    sm_str, doRandom, canonical, kekuleSmiles, allHsExplicit = args
    try:
        mol = Chem.MolFromSmiles(sm_str)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, doRandom=doRandom,
                               canonical=canonical,
                               kekuleSmiles=kekuleSmiles,
                               allHsExplicit=allHsExplicit)
    except:
        return None



In [3]:
df = pd.read_csv('chembl_33.csv')
sm_strs = list(df["rdkit_can_iso_smiles"])
sm_strs = convert_smiles_list(sm_strs, doRandom=True, canonical=False, kekuleSmiles=False, allHsExplicit=False)
jsondata = parallelize_processing(sm_strs)
outputfile = "random_smiles.jsonl"
with open(outputfile, 'w+') as out:
    for item in jsondata:
        out.write(json.dumps(item) + '\n')


Number of cores:  64


100%|██████████| 2330226/2330226 [05:02<00:00, 7694.48it/s]


In [3]:
df = pd.read_csv('chembl_33.csv')
sm_strs = list(df["rdkit_can_iso_smiles"])
sm_strs = convert_smiles_list(sm_strs, doRandom=False, canonical=True, kekuleSmiles=False, allHsExplicit=False)
jsondata = parallelize_processing(sm_strs)
outputfile = "canonical_smiles.jsonl"
with open(outputfile, 'w+') as out:
    for item in jsondata:
        out.write(json.dumps(item) + '\n')

Number of cores:  64


100%|██████████| 2240565/2240565 [05:09<00:00, 7249.46it/s]
